In [1]:
import Pkg
Pkg.activate("/home/masha/Phylotrajectories.jl")
# Pkg.add("NearestNeighbors")
using Phylotrajectories
using MolecularEvolution, Phylo, Plots, DataFrames, Statistics, Distributions, Random, CSV, StatsBase
using NearestNeighbors

  Activating project at `~/Phylotrajectories.jl`
[ Info: Precompiling Phylotrajectories [85c2bcf0-676d-4e10-9b42-7fb9c3e18bdd] (cache misses: include_dependency fsize change (2), wrong dep version loaded (2))
ERROR: Method overwriting is not permitted during Module precompilation. Use `__precompile__(false)` to opt-out of precompilation.
┌ Info: Skipping precompilation due to precompilable error. Importing Phylotrajectories [85c2bcf0-676d-4e10-9b42-7fb9c3e18bdd].
└   exception = Error when precompiling module, potentially caused by a __precompile__(false) declaration in the module.


In [2]:
using Random
# number_seed = 11
# Random.seed!(number_seed)

# Data

In [3]:
dataframe = CSV.read("/home/masha/PhyloTraj_paper/Final_notebooks/HDM/HDM_PCA.csv", DataFrame);
first(dataframe, 3)

Row,Column1,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,cell_types,Clone_Index,Treg_clustering
,String31,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,String15,Float64?,String15
1,AAACCTGAGAGGTTGC-1,-0.743021,-0.161295,0.571917,-0.907108,-0.132122,-0.398717,1.75852,-0.672198,-6.13114,-1.25123,-0.979772,1.68702,0.348738,0.192443,0.168332,-0.867778,0.767729,-0.309157,1.00148,-1.81584,0.63798,0.740046,-0.0351786,0.0485113,-0.369457,-0.236192,0.15176,0.314136,0.18317,-1.13165,-0.0620039,-0.689703,0.44235,-0.349973,0.125049,0.649964,2.0409,1.09307,0.698685,0.378015,0.0328321,1.06667,0.0101529,-0.160858,-0.718326,0.0686341,0.192795,-0.117315,0.235542,-0.0350318,IFN_response,1.0,IFN_response
2,AAACCTGAGGCGTACA-1,-0.540192,-0.434387,-1.26175,-0.01882,-0.168852,1.59527,0.334036,-0.0268919,-2.09923,0.322771,-0.932725,-1.42974,1.10001,0.221676,-0.340601,0.603578,-0.585283,0.627855,-0.568175,2.0686,0.488223,-0.549456,-0.0235427,0.201359,0.0218517,0.00661387,0.35767,-0.106375,0.909044,-0.052523,0.431464,-0.238979,-0.322608,-0.576283,-0.13646,0.104227,-0.674261,0.775924,-0.197462,-0.133992,-0.115407,-1.18924,0.724119,0.0798255,-0.245012,0.379537,-0.958801,0.397646,-0.695144,-0.57564,Th2_2,15.0,Th2_2
3,AAACCTGAGTGACTCT-1,-0.801896,0.307503,0.848741,-0.731001,-0.525557,-5.00545,-3.56763,0.614583,6.42511,1.82338,1.56302,-2.47575,-1.09148,-0.469359,-0.149741,0.405194,-0.16614,-0.24306,0.544713,-0.893847,-0.560223,0.676202,-0.228279,-0.503974,0.69404,-0.0138968,-0.649446,-0.352119,-0.366759,1.07418,-0.136863,0.104869,0.257507,0.612416,0.0906635,-0.552969,-1.94516,-0.621342,-0.624642,-0.50769,-0.0333419,-2.16899,0.291516,-0.995532,0.667141,-0.523767,0.325903,0.0630682,0.404908,-0.472331,Tregs,1.0,Tregs-2


In [4]:
countmap(dataframe[:,:cell_types])

Dict{String15, Int64} with 7 entries:
  "Naive"         => 319
  "Th2_2"         => 1356
  "Act_circ"      => 1099
  "IFN_response"  => 635
  "Proliferating" => 99
  "Th2"           => 1174
  "Tregs"         => 1611

In [5]:
countmap(dataframe[:,:Treg_clustering])

Dict{String15, Int64} with 10 entries:
  "Tregs-0"       => 474
  "Naive"         => 319
  "Th2_2"         => 1356
  "Act_circ"      => 1099
  "IFN_response"  => 635
  "Tregs-2"       => 370
  "Proliferating" => 99
  "Th2"           => 1174
  "Tregs-1"       => 462
  "Tregs-3"       => 305

In [23]:
dataframe_da = CSV.read("/home/masha/PhyloTraj_paper/Final_notebooks/DA/DA_PCA.csv", DataFrame);
first(dataframe_da, 3)

Row,Column1,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,cell_types,Clone_Index
,String31,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,String15,Float64
1,AAACCTGAGATAGCAT-1,8.02168,0.779392,-0.739466,0.621726,0.686227,-1.44744,1.7491,2.11345,-0.694238,2.5425,-2.5893,-3.86114,0.072077,0.419448,-0.696821,-0.654303,0.0319137,-2.01993,-3.47315,-0.986872,1.50426,-0.343811,0.31069,-1.28058,-1.59087,-0.0859381,-0.108159,0.674059,0.0153345,1.37366,0.0866118,0.469092,-0.781788,-0.567175,-0.762212,-0.140327,0.337949,0.190636,1.3373,0.99163,0.596855,0.360753,-1.75199,-0.805596,-0.942056,1.36556,0.391859,0.640593,-1.00092,1.373,Tfh,5.0
2,AAACCTGAGATCTGCT-1,14.7709,1.00954,-1.17397,0.392222,-0.0772328,-2.99688,0.295204,-0.684478,-0.0032549,4.81161,6.23294,5.23171,-0.224343,-0.91814,0.380596,0.503202,0.27633,-0.653496,-1.13247,-1.04253,-0.813091,-0.0653832,2.03199,-0.612474,-0.152003,0.499673,0.581591,-0.986291,0.655663,-0.263486,-0.782158,1.11487,0.838002,0.263031,0.946218,-1.57247,0.196065,0.0417474,-0.582084,-0.881861,-0.0873494,0.567165,0.218192,-0.0703837,0.446458,-0.460837,-1.18174,-0.130091,0.0654431,-0.297735,Naive,1.0
3,AAACCTGAGGCTACGA-1,6.0804,1.10918,-2.90166,0.933461,1.3846,-0.0105417,-1.90898,-3.9852,2.03418,-0.968088,-0.426979,2.40498,-0.230381,-0.821857,-0.344687,0.876695,-0.627075,0.154487,-0.980357,-0.527932,-1.04755,-1.47657,2.94594,-0.162834,-1.72091,-0.290019,0.291038,-1.14392,1.95774,-0.651856,-0.630947,1.18653,-0.623164,0.409354,-0.0635891,-0.272789,-0.275123,0.216314,-0.0422073,-0.187686,0.102832,0.263859,0.554536,0.190635,0.0673747,0.645683,0.609675,1.21165,1.27333,-0.0460539,Act_circ,1.0


In [7]:
countmap(dataframe_da[:,:cell_types])

Dict{String15, Int64} with 11 entries:
  "Act_circ"      => 597
  "Act2"          => 759
  "Th17"          => 1012
  "Th_CTL_like"   => 220
  "IFN_response"  => 51
  "Proliferating" => 20
  "Tfh"           => 566
  "Th2"           => 884
  "Tregs"         => 375
  "Naive"         => 354
  "Act3"          => 262

# Functions

In [8]:
function knn_neighbors(coords, k)
    # Build kd-tree for efficient nearest neighbor search
    kdtree = NearestNeighbors.KDTree(coords)
    
    # Find k nearest neighbors for each point
    idxs, _ = knn(kdtree, coords, k, true)  # k+1 because point itself is included

    return idxs
end

function MaximumCloneIdx(df, list_idx)
    return maximum(replace(df[list_idx, :Clone_Index], missing=>0))
end

function KnnMaxCloneIdx(df, k)
    neighbors = knn_neighbors(Matrix(df[:, 2:16])', k)
    values = map(n-> MaximumCloneIdx(df, n), neighbors)

    return values
end

# function ShuffleDf(df, N)
#     tmp = copy(df)
#     tmp[:, "Clone_Index"] = shuffle(tmp[:,:Clone_Index]);
#     tmp_values = KnnMaxCloneIdx(tmp, N);

#     return var(tmp_values)
# end

function ShuffleDf(df, N; rng=Random.default_rng())
    tmp = copy(df)
    tmp[!, :Clone_Index] = shuffle(rng, tmp[!, :Clone_Index])
    tmp_values = KnnMaxCloneIdx(tmp, N)

    return var(tmp_values)
end

ShuffleDf (generic function with 1 method)

In [9]:
function permutation_var_test(df, k; B=1000, rng=Random.default_rng())
    values_obs = KnnMaxCloneIdx(df, k)
    observed_var = var(values_obs)

    shuffled_vars = [ShuffleDf(df, k; rng=rng) for _ in 1:B]

    null_mean = mean(shuffled_vars)
    null_sd = std(shuffled_vars)

    # One-sided permutation p-value:
    # H1: observed variance is larger than expected under shuffled clone labels.
    pval = count(x -> x >= observed_var, shuffled_vars) / (B + 1)

    zscore =
        null_sd == 0 ?
        (observed_var == null_mean ? 0.0 : Inf * sign(observed_var - null_mean)) :
        (observed_var - null_mean) / null_sd

    effect_ratio = null_mean == 0 ? NaN : observed_var / null_mean
    q95 = quantile(shuffled_vars, 0.95)
    q99 = quantile(shuffled_vars, 0.99)

    return (
        values_obs = values_obs,
        observed_var = observed_var,
        shuffled_vars = shuffled_vars,
        null_mean = null_mean,
        null_sd = null_sd,
        pval = pval,
        zscore = zscore,
        effect_ratio = effect_ratio,
        q95 = q95,
        q99 = q99
    )
end


permutation_var_test (generic function with 1 method)

In [10]:
function holm_bonferroni_adjust(pvals::AbstractVector)
    m = length(pvals)
    order = sortperm(pvals)
    adjusted_pvals = similar(float.(pvals))

    running_max = 0.0

    for rank in 1:m
        i = order[rank]

        # Holm-Bonferroni step-down adjustment
        adjusted = (m - rank + 1) * pvals[i]

        # Enforce monotonicity of adjusted p-values
        running_max = max(running_max, adjusted)

        adjusted_pvals[i] = min(running_max, 1.0)
    end

    return adjusted_pvals
end

holm_bonferroni_adjust (generic function with 1 method)

# Make Stat Plots

## HDM

In [11]:
# before split 

dataset = "HDM";
path = "/home/masha/CLEAN_PhyloTrajectory_paper/final_images_for_paper/stat_test";

In [12]:
cell_types = unique(dataframe[:, :cell_types])

results = DataFrame(
    cell_type = String[],
    n_cells = Int[],
    observed_var = Float64[],
    null_mean = Float64[],
    null_sd = Float64[],
    pvalue = Float64[],
    zscore = Float64[],
    effect_ratio = Float64[],
    q95 = Float64[],
    q99 = Float64[]
)

tmp_plots = []

B = 1000
k = 50
rng = MersenneTwister(123)

for ct in cell_types
    println(ct)

    df = dataframe[dataframe.cell_types .== ct, :]

    # Optional guard for tiny cell-type groups.
    if nrow(df) < 3
        @warn "Skipping $ct because it has too few cells."
        continue
    end

    test = permutation_var_test(df, k; B=B, rng=rng)

    df[!, :KnnMaxCloneIdx] = test.values_obs

    push!(results, (
        cell_type = string(ct),
        n_cells = nrow(df),
        observed_var = test.observed_var,
        null_mean = test.null_mean,
        null_sd = test.null_sd,
        pvalue = test.pval,
        zscore = test.zscore,
        effect_ratio = test.effect_ratio,
        q95 = test.q95,
        q99 = test.q99
    ))

    plt = histogram(
        test.shuffled_vars,
        bins = 100,
        xtickfontsize = 16,
        ytickfontsize = 16,
        grid = false,
        color = :grey,
        fillalpha = 0.6,
        linecolor = :grey,
        alpha = 0,
        legend = false
    )

    vline!(plt, [test.observed_var], linewidth = 3, color = :red)

    title!(
        plt,
        "$ct  p=$(round(test.pval, digits=4))  z=$(round(test.zscore, digits=2))"
    )
    savefig(plt, "$(path)/$(dataset)_$(ct)_check_clustering_v3.pdf");
    push!(tmp_plots, plt)
end

IFN_response
Th2_2
Tregs
Act_circ
Th2
Naive
Proliferating


In [13]:
results

Row,cell_type,n_cells,observed_var,null_mean,null_sd,pvalue,zscore,effect_ratio,q95,q99
,String,Int64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64
1,IFN_response,635,2663.04,1800.8,346.373,0.004995,2.48933,1.47881,2356.93,2569.89
2,Th2_2,1356,277.551,151.291,59.1,0.034965,2.13637,1.83455,259.791,323.174
3,Tregs,1611,1350.89,233.713,75.0271,0.0,14.8903,5.78013,373.642,447.143
4,Act_circ,1099,1700.86,1694.47,313.901,0.501499,0.020363,1.00377,2198.95,2391.49
5,Th2,1174,329.895,275.372,108.471,0.266733,0.502653,1.198,466.755,599.326
6,Naive,319,97.7448,113.278,23.2681,0.743257,-0.667597,0.862872,151.24,167.415
7,Proliferating,99,168.102,538.974,571.982,0.605395,-0.648398,0.311893,1600.9,2300.78


In [14]:
holm_bonferroni_adjust(results.pvalue)

7-element Vector{Float64}:
 0.02997002997002997
 0.17482517482517484
 0.0
 1.0
 1.0
 1.0
 1.0

In [15]:
results[!, :qvalue_BonfHolm] = holm_bonferroni_adjust(results.pvalue);

results[!, :significant_FDR_05] = results.qvalue_BonfHolm .< 0.05;
results[!, :significant_FDR_01] = results.qvalue_BonfHolm .< 0.01;

results

Row,cell_type,n_cells,observed_var,null_mean,null_sd,pvalue,zscore,effect_ratio,q95,q99,qvalue_BonfHolm,significant_FDR_05,significant_FDR_01
,String,Int64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Bool,Bool
1,IFN_response,635,2663.04,1800.8,346.373,0.004995,2.48933,1.47881,2356.93,2569.89,0.02997,true,false
2,Th2_2,1356,277.551,151.291,59.1,0.034965,2.13637,1.83455,259.791,323.174,0.174825,false,false
3,Tregs,1611,1350.89,233.713,75.0271,0.0,14.8903,5.78013,373.642,447.143,0.0,true,true
4,Act_circ,1099,1700.86,1694.47,313.901,0.501499,0.020363,1.00377,2198.95,2391.49,1.0,false,false
5,Th2,1174,329.895,275.372,108.471,0.266733,0.502653,1.198,466.755,599.326,1.0,false,false
6,Naive,319,97.7448,113.278,23.2681,0.743257,-0.667597,0.862872,151.24,167.415,1.0,false,false
7,Proliferating,99,168.102,538.974,571.982,0.605395,-0.648398,0.311893,1600.9,2300.78,1.0,false,false


In [16]:
CSV.write("$(path)/$(dataset)_clustering_test_results_BonferroniHolm.csv", results, index=false, sep="\t")

"/home/masha/CLEAN_PhyloTrajectory_paper/final_images_for_paper/stat_test/HDM_clustering_test_results_BonferroniHolm.csv"

## HDM_split

In [17]:
# after split

dataset = "HDM_split";
path = "/home/masha/CLEAN_PhyloTrajectory_paper/final_images_for_paper/stat_test";

In [19]:
cell_types = ["Tregs-0", "Tregs-1", "Tregs-2", "Tregs-3"];

results = DataFrame(
    cell_type = String[],
    n_cells = Int[],
    observed_var = Float64[],
    null_mean = Float64[],
    null_sd = Float64[],
    pvalue = Float64[],
    zscore = Float64[],
    effect_ratio = Float64[],
    q95 = Float64[],
    q99 = Float64[]
)

tmp_plots = []

B = 1000
k = 50
rng = MersenneTwister(123)

for ct in cell_types
    println(ct)

    df = dataframe[dataframe.Treg_clustering .== ct, :]

    # Optional guard for tiny cell-type groups.
    if nrow(df) < 3
        @warn "Skipping $ct because it has too few cells."
        continue
    end

    test = permutation_var_test(df, k; B=B, rng=rng)

    df[!, :KnnMaxCloneIdx] = test.values_obs

    push!(results, (
        cell_type = string(ct),
        n_cells = nrow(df),
        observed_var = test.observed_var,
        null_mean = test.null_mean,
        null_sd = test.null_sd,
        pvalue = test.pval,
        zscore = test.zscore,
        effect_ratio = test.effect_ratio,
        q95 = test.q95,
        q99 = test.q99
    ))

    plt = histogram(
        test.shuffled_vars,
        bins = 100,
        xtickfontsize = 16,
        ytickfontsize = 16,
        grid = false,
        color = :grey,
        fillalpha = 0.6,
        linecolor = :grey,
        alpha = 0,
        legend = false
    )

    vline!(plt, [test.observed_var], linewidth = 3, color = :red)

    title!(
        plt,
        "$ct  p=$(round(test.pval, digits=4))  z=$(round(test.zscore, digits=2))"
    )
    savefig(plt, "$(path)/$(dataset)_$(ct)_check_clustering_v3.pdf");
    push!(tmp_plots, plt)
end

Tregs-0
Tregs-1
Tregs-2
Tregs-3


In [26]:
results

Row,cell_type,n_cells,observed_var,null_mean,null_sd,pvalue,zscore,effect_ratio,q95,q99
,String,Int64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64
1,Tregs-0,474,41.238,87.0899,82.5585,0.656344,-0.555386,0.473511,235.195,390.561
2,Tregs-1,462,25.1609,15.7031,22.5156,0.190809,0.420057,1.60229,59.8075,104.074
3,Tregs-2,370,3460.3,3000.28,804.829,0.324675,0.571571,1.15332,4132.7,4425.72
4,Tregs-3,305,417.588,593.332,254.656,0.737263,-0.690124,0.703802,1048.44,1316.14


In [20]:
holm_bonferroni_adjust(results.pvalue)

4-element Vector{Float64}:
 1.0
 0.7592407592407593
 0.971028971028971
 1.0

In [21]:
results[!, :qvalue_BonfHolm] = holm_bonferroni_adjust(results.pvalue);

results[!, :significant_FDR_05] = results.qvalue_BonfHolm .< 0.05;
results[!, :significant_FDR_01] = results.qvalue_BonfHolm .< 0.01;

results

Row,cell_type,n_cells,observed_var,null_mean,null_sd,pvalue,zscore,effect_ratio,q95,q99,qvalue_BonfHolm,significant_FDR_05,significant_FDR_01
,String,Int64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Bool,Bool
1,Tregs-0,474,41.238,87.0899,82.5585,0.655345,-0.555386,0.473511,235.195,390.561,1.0,false,false
2,Tregs-1,462,25.1609,15.7031,22.5156,0.18981,0.420057,1.60229,59.8075,104.074,0.759241,false,false
3,Tregs-2,370,3460.3,3000.28,804.829,0.323676,0.571571,1.15332,4132.7,4425.72,0.971029,false,false
4,Tregs-3,305,417.588,593.332,254.656,0.736264,-0.690124,0.703802,1048.44,1316.14,1.0,false,false


In [22]:
CSV.write("$(path)/$(dataset)_clustering_test_results_BonferroniHolm.csv", results, index=false, sep="\t")

"/home/masha/CLEAN_PhyloTrajectory_paper/final_images_for_paper/stat_test/HDM_split_clustering_test_results_BonferroniHolm.csv"

## Dog Allergen

In [24]:
dataset = "DA";
path = "/home/masha/CLEAN_PhyloTrajectory_paper/final_images_for_paper/stat_test";

In [26]:
cell_types = [ "Tfh",
 "Naive",
 "Act_circ",
 "Th2",
 "Tregs",
 "Th_CTL_like",
 "Act2",
 "Act3",
 "Th17",
 "IFN_response",
# "Proliferating"
 ]

results = DataFrame(
    cell_type = String[],
    n_cells = Int[],
    observed_var = Float64[],
    null_mean = Float64[],
    null_sd = Float64[],
    pvalue = Float64[],
    zscore = Float64[],
    effect_ratio = Float64[],
    q95 = Float64[],
    q99 = Float64[]
)

tmp_plots = []

B = 1000
k = 50
rng = MersenneTwister(123)

for ct in cell_types
    println(ct)

    df = dataframe_da[dataframe_da.cell_types .== ct, :]

    # Optional guard for tiny cell-type groups.
    if nrow(df) < 3
        @warn "Skipping $ct because it has too few cells."
        continue
    end

    test = permutation_var_test(df, k; B=B, rng=rng)

    df[!, :KnnMaxCloneIdx] = test.values_obs

    push!(results, (
        cell_type = string(ct),
        n_cells = nrow(df),
        observed_var = test.observed_var,
        null_mean = test.null_mean,
        null_sd = test.null_sd,
        pvalue = test.pval,
        zscore = test.zscore,
        effect_ratio = test.effect_ratio,
        q95 = test.q95,
        q99 = test.q99
    ))

    plt = histogram(
        test.shuffled_vars,
        bins = 100,
        xtickfontsize = 16,
        ytickfontsize = 16,
        grid = false,
        color = :grey,
        fillalpha = 0.6,
        linecolor = :grey,
        alpha = 0,
        legend = false
    )

    vline!(plt, [test.observed_var], linewidth = 3, color = :red)

    title!(
        plt,
        "$ct  p=$(round(test.pval, digits=4))  z=$(round(test.zscore, digits=2))"
    )
    savefig(plt, "$(path)/$(dataset)_$(ct)_check_clustering_v3.pdf");
    push!(tmp_plots, plt)
end

Tfh
Naive
Act_circ
Th2
Tregs
Th_CTL_like
Act2
Act3
Th17
IFN_response


In [27]:
results

Row,cell_type,n_cells,observed_var,null_mean,null_sd,pvalue,zscore,effect_ratio,q95,q99
,String,Int64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64
1,Tfh,566,3514.03,3529.86,493.467,0.541459,-0.0320883,0.995514,4313.1,4607.56
2,Naive,354,0.330212,0.569378,0.113438,0.989011,-2.10834,0.579953,0.757582,0.813945
3,Act_circ,597,1344.94,727.65,282.271,0.020979,2.18687,1.84833,1243.24,1577.87
4,Th2,884,296.362,182.273,78.6223,0.0849151,1.4511,1.62592,328.995,396.955
5,Tregs,375,154.132,105.775,84.374,0.220779,0.573134,1.45717,270.703,389.834
6,Th_CTL_like,220,6490.07,5939.31,1180.16,0.328671,0.466682,1.09273,7809.1,8467.77
7,Act2,759,612.445,304.74,145.544,0.02997,2.11417,2.00973,577.789,758.671
8,Act3,262,313.117,144.37,130.988,0.104895,1.28826,2.16884,397.37,614.981
9,Th17,1012,727.934,611.441,209.437,0.287712,0.556223,1.19052,994.743,1117.0


In [28]:
holm_bonferroni_adjust(results.pvalue)

10-element Vector{Float64}:
 1.0
 1.0
 0.2097902097902098
 0.6793206793206793
 1.0
 1.0
 0.26973026973026976
 0.7342657342657343
 1.0
 1.0

In [29]:
results[!, :qvalue_BonfHolm] = holm_bonferroni_adjust(results.pvalue);

results[!, :significant_FDR_05] = results.qvalue_BonfHolm .< 0.05;
results[!, :significant_FDR_01] = results.qvalue_BonfHolm .< 0.01;

results

Row,cell_type,n_cells,observed_var,null_mean,null_sd,pvalue,zscore,effect_ratio,q95,q99,qvalue_BonfHolm,significant_FDR_05,significant_FDR_01
,String,Int64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Bool,Bool
1,Tfh,566,3514.03,3529.86,493.467,0.541459,-0.0320883,0.995514,4313.1,4607.56,1.0,false,false
2,Naive,354,0.330212,0.569378,0.113438,0.989011,-2.10834,0.579953,0.757582,0.813945,1.0,false,false
3,Act_circ,597,1344.94,727.65,282.271,0.020979,2.18687,1.84833,1243.24,1577.87,0.20979,false,false
4,Th2,884,296.362,182.273,78.6223,0.0849151,1.4511,1.62592,328.995,396.955,0.679321,false,false
5,Tregs,375,154.132,105.775,84.374,0.220779,0.573134,1.45717,270.703,389.834,1.0,false,false
6,Th_CTL_like,220,6490.07,5939.31,1180.16,0.328671,0.466682,1.09273,7809.1,8467.77,1.0,false,false
7,Act2,759,612.445,304.74,145.544,0.02997,2.11417,2.00973,577.789,758.671,0.26973,false,false
8,Act3,262,313.117,144.37,130.988,0.104895,1.28826,2.16884,397.37,614.981,0.734266,false,false
9,Th17,1012,727.934,611.441,209.437,0.287712,0.556223,1.19052,994.743,1117.0,1.0,false,false


In [30]:
CSV.write("$(path)/$(dataset)_clustering_test_results_BonferroniHolm.csv", results, index=false, sep="\t")

"/home/masha/CLEAN_PhyloTrajectory_paper/final_images_for_paper/stat_test/DA_clustering_test_results_BonferroniHolm.csv"